# Colab STaR Rationale Data Builder

This notebook builds answer-conditioned STaR-style rationale data for the two hardest families: cipher and bit manipulation. It is the follow-up after free-form RFT-style sampling failed for cipher in exp14/exp18.

The model is frozen during generation. For each training row, the known gold answer is supplied so the model can write a compact rationale that reaches that answer. Raw generations are saved before filtering. Accepted rationales can later be mixed into a normal SFT training notebook.

## Method

1. Start from official `train.csv`, not from an existing trace file.
2. Exclude public `test.csv` IDs when that file is present.
3. Infer the problem family for reporting.
4. Filter source rows to `cipher` and `bit_manipulation` only.
5. Default to a balanced 32-question probe before full generation.
6. Give the frozen model the question and known `gold_answer`.
7. Ask for a short mechanical rationale ending with exactly one boxed final answer.
8. Save every raw rationale before filtering.
9. Accept only rationales whose boxed answer matches gold and pass quality gates.
10. Select one accepted rationale per row.
11. Write a trainable CSV with schema `id,question,trace,gold_answer` plus diagnostics.

This is STaR-style data construction, not RFT: the answer is supplied because open rejection sampling had near-zero accepted cipher coverage. The first gate is rationale quality on 32 rows. Do not scale to full generation unless the probe outputs are compact and mechanically plausible.

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "vllm", "torch", "torchvision", "torchaudio", "xformers", "triton"
], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

subprocess.run([
    sys.executable,
    "-m",
    "uv",
    "pip",
    "install",
    "--system",
    "-U",
    "vllm==0.12.0",
    "--torch-backend=auto",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "show", "torch", "vllm"], check=False)

# Force kernel restart so the newly installed torch/vLLM binaries are loaded cleanly.
os.kill(os.getpid(), 9)

In [ ]:
import os
# Must be set before the first vLLM import in this notebook.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import torch
import vllm
from vllm import LLM, SamplingParams

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("vllm:", vllm.__version__)
print("VLLM_ENABLE_V1_MULTIPROCESSING:", os.environ.get("VLLM_ENABLE_V1_MULTIPROCESSING"))

## 1. Config


In [ ]:
# Change only this cell for a new STaR rationale build.
EXPERIMENT_NAME = "exp19_cipher_bit_star_rationale_probe_v4"
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# vLLM path. Keep this notebook generation-only; training notebooks still build the LoRA adapter.
GENERATION_BACKEND = "vllm"
VLLM_DTYPE = "bfloat16"
VLLM_TENSOR_PARALLEL_SIZE = 1
VLLM_GPU_MEMORY_UTILIZATION = 0.92
VLLM_ENFORCE_EAGER = False
VLLM_TRUST_REMOTE_CODE = True

# Keep the first probe small and balanced. Set PROBE_ONLY=False only after inspecting outputs.
TARGET_SAMPLE_FAMILIES = {"cipher", "bit_manipulation"}
PROBE_ONLY = True
PROBE_TOTAL_ROWS = 32
PROBE_ROWS_PER_FAMILY_CAP = PROBE_TOTAL_ROWS // len(TARGET_SAMPLE_FAMILIES)
ROW_LIMIT = None
ROWS_PER_FAMILY_CAP = None
CANDIDATES_PER_ROW = 1
GENERATION_BATCH_SIZE = 32
SAVE_RAW_EVERY_BATCHES = 1
RUN_GENERATION_SPEED_BENCHMARK = True
BENCHMARK_ROWS = 16
BENCHMARK_MAX_NEW_TOKENS = 64

MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 256
ALLOW_PROMPT_TRUNCATION = False
TEMPERATURE = 0.4
TOP_P = 0.9
REPETITION_PENALTY = 1.03

# Acceptance gates. Start strict; relax only with evidence from the probe.
REQUIRE_EXACTLY_ONE_BOX = True
REJECT_MAX_TOKEN_HITS = True
REJECT_TEXT_AFTER_BOX = True
REQUIRE_FAMILY_TRACE_SIGNAL = True
MAX_ACCEPTED_CHARS = 1400
MAX_ACCEPTED_PER_FAMILY = None
INCLUDE_BOXED_FALLBACKS = False

RANDOM_SEED = 20260606

## 2. Paths


In [ ]:
from pathlib import Path
from google.colab import drive

PROJECT_ROOT = Path("/content/nemotron_challenge")
CONTENT_DIR = Path("/content")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts")
INPUT_DIR = PROJECT_ROOT / "data" / "input"

TRAIN_CSV_CANDIDATES = [
    CONTENT_DIR / "train.csv",
    INPUT_DIR / "official" / "train.csv",
]
TEST_CSV_CANDIDATES = [
    CONTENT_DIR / "test.csv",
    INPUT_DIR / "official" / "test.csv",
]

OUTPUT_DIR = DRIVE_PROJECT_ROOT / "outputs" / EXPERIMENT_NAME
RAW_CANDIDATES_CSV = OUTPUT_DIR / "star_rationale_raw_generations.csv"
ACCEPTED_CANDIDATES_CSV = OUTPUT_DIR / "star_rationale_accepted_generations.csv"
ACCEPTANCE_SUMMARY_CSV = OUTPUT_DIR / "star_rationale_summary.csv"
TRACE_DATASET_CSV = OUTPUT_DIR / f"trace_{EXPERIMENT_NAME}.csv"
GENERATION_SPEED_BENCHMARK_CSV = OUTPUT_DIR / "generation_speed_benchmark.csv"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"

drive.mount("/content/drive", force_remount=False)
if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Google Drive is not mounted at /content/drive/MyDrive")

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive output root:", DRIVE_PROJECT_ROOT)
print("output_dir:", OUTPUT_DIR)


## 3. Imports


In [ ]:

import os
import subprocess
import sys
# Notebook-safe vLLM path: avoid spawning a separate EngineCore process after
# Colab/Jupyter or diagnostics have already touched CUDA.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import json
import random
import re
import time
from dataclasses import asdict, dataclass
from decimal import Decimal, InvalidOperation

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

print("generation backend:", GENERATION_BACKEND)
print("VLLM_ENABLE_V1_MULTIPROCESSING:", os.environ.get("VLLM_ENABLE_V1_MULTIPROCESSING"))
if GENERATION_BACKEND != "vllm":
    raise ValueError("Notebook 12 is intentionally vLLM-only. Set GENERATION_BACKEND='vllm'.")

random.seed(RANDOM_SEED)

## 4. Load Official Train Data


In [ ]:
import pandas as pd
from IPython.display import display


def first_existing_path(candidates: list[Path], label: str) -> Path:
    """Return the first existing required input path, or stop with all allowed locations.

    Args:
        candidates: Ordered file paths to try, usually Colab upload path first and repo fallback second.
        label: Human-readable file name used in status/error messages.

    Returns:
        The first path that exists on disk.
    """
    for path in candidates:
        if path.exists():
            print(f"using {label}: {path}")
            return path
    formatted = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(f"Missing {label}. Put it in one of these locations:\n{formatted}")


def optional_existing_path(candidates: list[Path], label: str) -> Path | None:
    """Return an optional input path when present; absence should not stop the run.

    Args:
        candidates: Ordered file paths to try.
        label: Human-readable file name used in status messages.

    Returns:
        The first existing path, or None when the optional file is unavailable.
    """
    for path in candidates:
        if path.exists():
            print(f"using {label}: {path}")
            return path
    print(f"optional {label} not found; no public sanity ID exclusion applied")
    return None


TRAIN_CSV_PATH = first_existing_path(TRAIN_CSV_CANDIDATES, "train.csv")
train = pd.read_csv(TRAIN_CSV_PATH, dtype=str).fillna("")

expected_cols = ["id", "question", "gold_answer"]
if list(train.columns) != expected_cols:
    raise ValueError(f"train.csv must have columns {expected_cols}, got {list(train.columns)}")

TEST_CSV_PATH = optional_existing_path(TEST_CSV_CANDIDATES, "test.csv")
if TEST_CSV_PATH is not None:
    test = pd.read_csv(TEST_CSV_PATH, dtype=str).fillna("")
    if "prompt" in test.columns and "question" not in test.columns:
        test = test.rename(columns={"prompt": "question"})
    if "id" not in test.columns:
        raise ValueError(f"test.csv must include id, got {list(test.columns)}")
    public_test_ids = set(test["id"].astype(str))
    before_rows = len(train)
    train = train[~train["id"].astype(str).isin(public_test_ids)].reset_index(drop=True)
    print("excluded public sanity IDs:", before_rows - len(train))

print("train:", train.shape)
display(train.head())

## 5. Family Inference And Verifier


In [ ]:
def infer_family(prompt: str) -> str:
    """Infer the coarse puzzle family from prompt text for sampling and acceptance reports.

    Args:
        prompt: Original puzzle question text.

    Returns:
        Family label such as cipher, bit_manipulation, unit_conversion, or unknown.
    """
    text = str(prompt).lower()
    if "bit manipulation" in text or "8-bit binary" in text:
        return "bit_manipulation"
    if "secret encryption" in text or "decrypt" in text:
        return "cipher"
    if "numeral system" in text:
        return "numeral"
    if "unit" in text and "convert" in text:
        return "unit_conversion"
    if "gravitational constant" in text or "falling distance" in text:
        return "gravity"
    if "transformation rules is applied to equations" in text or "determine the result for" in text:
        return "equation_symbolic"
    return "unknown"


def normalize_answer(value) -> str:
    """Normalize lightweight answer formatting before exact or Decimal-based comparison.

    Args:
        value: Gold or predicted answer value from CSV/model output.

    Returns:
        A stripped one-line answer string with backtick wrappers and repeated whitespace removed.
    """
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".")


def extract_final_answer(text: str | None) -> str:
    r"""Extract the answer, preferring Kaggle-style last-brace boxed content when present.

    Args:
        text: Raw model completion.

    Returns:
        Extracted final answer text, or NOT_FOUND when the completion is empty.
    """
    if text is None:
        return "NOT_FOUND"

    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    matches = []
    for idx, match in enumerate(boxed_starts):
        start = match.end()
        end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind("}")
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [match.strip() for match in matches if match.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:\uFF1A]\s*([^\n]+)",
        r"final answer\s*[:\uFF1A]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return number_matches[-1]

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify_answer(stored_answer: str, predicted: str) -> bool:
    """Strictly verify candidate answers for training-data acceptance, not leaderboard scoring.

    Args:
        stored_answer: Gold answer from official train.csv.
        predicted: Extracted answer from one sampled completion.

    Returns:
        True only when answers match exactly after normalization, with Decimal equality for numeric strings.
    """
    stored_answer = normalize_answer(stored_answer).replace(",", "")
    predicted = normalize_answer(predicted).replace(",", "")
    if re.fullmatch(r"[01]+", stored_answer):
        return predicted.lower() == stored_answer.lower()
    if stored_answer.lower() == predicted.lower():
        return True
    try:
        return Decimal(stored_answer) == Decimal(predicted)
    except (InvalidOperation, ValueError):
        return False


def boxed_only_trace(gold_answer: str) -> str:
    """Build an explicit boxed fallback target when fallback rows are intentionally enabled.

    Args:
        gold_answer: Gold answer to place inside the boxed final answer.

    Returns:
        Minimal assistant completion containing exactly one boxed final answer.
    """
    return f"Final answer: \\boxed{{{str(gold_answer).strip()}}}"


def extract_last_boxed_content_and_tail(text: str) -> tuple[str, str, bool]:
    r"""Return last boxed content, trailing text after its closing brace, and closure status.

    Args:
        text: Raw model completion to inspect.

    Returns:
        A tuple of boxed content, text after the closing brace, and whether a closing brace was found.
    """
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if not boxed_starts:
        return "", text, False
    start = boxed_starts[-1].end()
    segment = text[start:]
    last_brace = segment.rfind("}")
    if last_brace == -1:
        return segment, "", False
    return segment[:last_brace], segment[last_brace + 1:], True


def has_family_trace_signal(raw_output: str, family: str) -> bool:
    """Reject lucky short answers that lack minimal family-specific reasoning markers.

    Args:
        raw_output: Raw sampled completion.
        family: Inferred puzzle family for the source row.

    Returns:
        True when the completion contains minimal family-relevant reasoning markers.
    """
    text = raw_output.lower()
    if family == "cipher":
        pair_pattern = r"\b[a-z]{2,}\s*(?:->|=>|\u2192| maps to )\s*[a-z]{2,}\b"
        align_line = next((line.strip() for line in text.splitlines() if line.strip().startswith("align:")), "")
        target_line = next((line.strip() for line in text.splitlines() if line.strip().startswith("target:")), "")
        align_pairs = re.findall(pair_pattern, align_line)
        target_pairs = re.findall(pair_pattern, target_line)
        final_words = re.findall(r"[a-z]+", normalize_answer(extract_final_answer(raw_output)).lower())
        return len(align_pairs) >= 1 and len(target_pairs) >= max(1, len(final_words))
    if family == "bit_manipulation":
        return bool(re.search(r"[01]{4,}", text)) and any(word in text for word in ["bit", "rule", "xor", "and", "or", "shift"])
    if family == "equation_symbolic":
        return any(word in text for word in ["rule", "transform", "substitution", "equation", "apply"])
    if family == "gravity":
        return bool(re.search(r"\bg\b|t\^2|0\.5|distance|round|constant", text))
    if family == "unit_conversion":
        return any(word in text for word in ["ratio", "convert", "conversion", "unit", "k="])
    if family == "numeral":
        return any(word in text for word in ["numeral", "convert", "value", "+"])
    return True


def trace_signal_score(raw_output: str, family: str) -> int:
    """Score accepted traces so selection prefers plausible reasoning over shortest text only.

    Args:
        raw_output: Raw sampled completion already passing correctness gates.
        family: Inferred puzzle family for the source row.

    Returns:
        Integer heuristic score; higher means more family-relevant trace signal.
    """
    text = raw_output.lower()
    keywords = {
        "cipher": ["->", "=>", "\u2192", "maps to", "target:", "cipher", "plain", "mapping", "align", "letter"],
        "bit_manipulation": ["bit", "rule", "xor", "and", "or", "shift", "binary"],
        "equation_symbolic": ["rule", "transform", "substitution", "equation", "apply"],
        "gravity": ["t^2", "0.5", "distance", "round", "constant"],
        "unit_conversion": ["ratio", "convert", "conversion", "unit", "k="],
        "numeral": ["numeral", "convert", "value", "+"],
    }.get(family, [])
    score = sum(1 for keyword in keywords if keyword in text)
    if family == "cipher":
        target_line = next((line.strip() for line in text.splitlines() if line.strip().startswith("target:")), "")
        score += min(4, len(re.findall(r"\b[a-z]{2,}\s*(?:->|=>|\u2192| maps to )\s*[a-z]{2,}\b", target_line)))
    score += int(len(raw_output) >= 120)
    score += int(raw_output.count("\n") >= 2)
    return score


def quality_gate(raw_output: str, hit_max_new_tokens: bool, family: str) -> tuple[bool, str]:
    """Apply non-answer quality gates and return a machine-readable rejection reason.

    Args:
        raw_output: Raw sampled completion.
        hit_max_new_tokens: Whether generation stopped only because MAX_NEW_TOKENS was reached.
        family: Inferred puzzle family used for trace-signal checks.

    Returns:
        A pair: gate_ok boolean and rejection reason such as ok, too_long, or text_after_box.
    """
    if REJECT_MAX_TOKEN_HITS and hit_max_new_tokens:
        return False, "max_token_hit"
    if REQUIRE_EXACTLY_ONE_BOX and raw_output.count("\\boxed{") != 1:
        return False, "boxed_count"
    _, tail, box_closed = extract_last_boxed_content_and_tail(raw_output)
    if raw_output.count("\\boxed{") and not box_closed:
        return False, "unclosed_box"
    if REJECT_TEXT_AFTER_BOX and tail.strip():
        return False, "text_after_box"
    if len(raw_output) > MAX_ACCEPTED_CHARS:
        return False, "too_long"
    if not raw_output.strip():
        return False, "empty"
    if REQUIRE_FAMILY_TRACE_SIGNAL and not has_family_trace_signal(raw_output, family):
        return False, "missing_trace_signal"
    return True, "ok"


train["family"] = train["question"].map(infer_family)
display(train["family"].value_counts().rename_axis("family").reset_index(name="rows"))


## 6. Choose Rows

This pass samples only cipher and bit manipulation. The other families keep the existing deterministic template or boxed-control path in downstream training; they are not sampled here and should not appear in the raw rationale CSV.


In [ ]:
def choose_source_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Create a reproducible hard-family subset for STaR rationale generation."""
    work = df[df["family"].isin(TARGET_SAMPLE_FAMILIES)].copy()
    if work.empty:
        raise ValueError(f"No rows found for TARGET_SAMPLE_FAMILIES={sorted(TARGET_SAMPLE_FAMILIES)}")
    work = work.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    if PROBE_ONLY:
        work = (
            work.groupby("family", group_keys=False)
            .head(int(PROBE_ROWS_PER_FAMILY_CAP))
            .reset_index(drop=True)
        )
    elif ROWS_PER_FAMILY_CAP is not None:
        work = (
            work.groupby("family", group_keys=False)
            .head(int(ROWS_PER_FAMILY_CAP))
            .reset_index(drop=True)
        )
    if ROW_LIMIT is not None:
        work = work.head(int(ROW_LIMIT)).reset_index(drop=True)
    return work.sort_values(["family", "id"]).reset_index(drop=True)


source_rows = choose_source_rows(train)
print("target sample families:", sorted(TARGET_SAMPLE_FAMILIES))
print("probe_only:", PROBE_ONLY)
print("source rows:", source_rows.shape)
display(source_rows["family"].value_counts().rename_axis("family").reset_index(name="rows"))
display(source_rows.head())

## 7. Answer-Conditioned Rationale Prompt

The prompt deliberately reveals the training-row gold answer. This is the STaR-style pivot after open rejection sampling produced no usable cipher data. The output must not say the answer was provided; it should read like a compact mechanical solution trace.

In [ ]:
SYSTEM_PROMPT = (
    "Write a very short training rationale. Use at most four short lines. "
    "Do not mention that the final answer was supplied. "
    "Use the required result inside the final boxed answer. "
    "No text after the boxed answer."
)

FAMILY_HINTS = {
    "bit_manipulation": (
        "Required boxed result: {answer}. Use this compact format only: "
        "Rule: one concise candidate bit rule. Check: one example check. "
        "Target: apply the rule to the target bits. Final: boxed result."
    ),
    "cipher": (
        "Required boxed result: {answer}. Use this compact format only: "
        "Align: one example as cipher_word -> plain_word. "
        "Target: show each target word as cipher_word -> plain_word, one pair per final word. "
        "Final: boxed result."
    ),
}


def build_candidate_prompt(question: str, family: str, gold_answer: str) -> str:
    """Build the exact answer-conditioned STaR prompt for rationale generation."""
    if family not in TARGET_SAMPLE_FAMILIES:
        raise ValueError(f"Notebook 12 should sample only {sorted(TARGET_SAMPLE_FAMILIES)}, got {family}")
    answer = normalize_answer(gold_answer)
    hint = FAMILY_HINTS[family].format(answer=answer)
    return (
        f"System:\n{SYSTEM_PROMPT}\n\n"
        f"User:\n{hint}\n\nPuzzle:\n{question}\n\n"
        "Assistant:\n"
    )


# Override the generic quality gate with STaR-specific leakage and contradiction checks.
_base_quality_gate = quality_gate

ANSWER_LEAK_PHRASES = [
    "given answer", "provided answer", "known answer", "gold answer",
    "the answer was provided", "we are told", "since the correct answer",
    "required boxed result", "required result", "answer was supplied",
    "expected output", "expected answer", "expected result",
    "produce final answer", "final answer exactly", "must output",
]

SELF_CONTRADICTION_PHRASES = [
    "wait need to check", "wait, need to check", "need to check",
    "does not match", "doesn't match", "not match", "mismatch",
    "incorrect", "is wrong", "rule is wrong", "which is not",
    "but the example shows", "scrap that", "too long", "messy",
    "contradict", "but expected", "but the expected",
]


def quality_gate(raw_output: str, hit_max_new_tokens: bool, family: str) -> tuple[bool, str]:
    ok, reason = _base_quality_gate(raw_output, hit_max_new_tokens, family)
    if not ok:
        return ok, reason
    low = str(raw_output).lower()
    if any(phrase in low for phrase in SELF_CONTRADICTION_PHRASES):
        return False, "self_contradiction"
    if any(phrase in low for phrase in ANSWER_LEAK_PHRASES):
        return False, "answer_leak_phrase"
    return True, "ok"


print(build_candidate_prompt(source_rows.iloc[0].question, source_rows.iloc[0].family, source_rows.iloc[0].gold_answer)[:1400])

## 8. Load vLLM Teacher

This cell loads the base model through vLLM for offline sampling throughput. It does not train and it does not package a Kaggle adapter. If vLLM cannot load this Nemotron model on the selected Colab GPU, that is useful backend evidence; do not silently fall back to the old Transformers path in this notebook.


In [ ]:
from contextlib import contextmanager


@contextmanager
def notebook_safe_suppress_stdout():
    yield


def patch_vllm_notebook_stdout() -> None:
    """Avoid vLLM 0.12 sys.stdout.fileno() failures under ipykernel."""
    import vllm.distributed.parallel_state as vllm_parallel_state
    import vllm.utils.system_utils as vllm_system_utils

    vllm_system_utils.suppress_stdout = notebook_safe_suppress_stdout
    vllm_parallel_state.suppress_stdout = notebook_safe_suppress_stdout
    print("patched vLLM suppress_stdout for notebook stdout")


patch_vllm_notebook_stdout()

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=VLLM_TRUST_REMOTE_CODE,
    dtype=VLLM_DTYPE,
    tensor_parallel_size=VLLM_TENSOR_PARALLEL_SIZE,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_SEQ_LENGTH + MAX_NEW_TOKENS,
    enforce_eager=VLLM_ENFORCE_EAGER,
    seed=RANDOM_SEED,
)

tokenizer = llm.get_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("vLLM model loaded")
print("generation backend:", GENERATION_BACKEND)
print("vLLM dtype:", VLLM_DTYPE)
print("max_model_len:", MAX_SEQ_LENGTH + MAX_NEW_TOKENS)


## 8B. Prompt Length Audit

For the current official train prompts, truncation should normally be unnecessary at `MAX_SEQ_LENGTH=1024`. This audit makes that assumption explicit. If the prompt builder changes later and prompts exceed the limit, stop and inspect rather than silently cutting evidence from the puzzle.


In [ ]:
def audit_prompt_lengths(rows: pd.DataFrame) -> pd.DataFrame:
    """Measure tokenized prompt lengths before candidate generation.

    Args:
        rows: Source rows that will be sampled, with question and family columns.

    Returns:
        Per-family prompt-token summary used to decide whether truncation is active.
    """
    prompts = [build_candidate_prompt(row.question, row.family, row.gold_answer) for row in rows.itertuples(index=False)]
    lengths = [len(tokenizer(prompt, add_special_tokens=True)["input_ids"]) for prompt in prompts]
    audited = rows[["id", "family"]].copy()
    audited["prompt_tokens"] = lengths
    summary = (
        audited.groupby("family")["prompt_tokens"]
        .agg(rows="count", min="min", median="median", p95=lambda s: s.quantile(0.95), max="max")
        .reset_index()
    )
    display(summary)
    too_long = audited[audited["prompt_tokens"] > MAX_SEQ_LENGTH]
    if not too_long.empty:
        display(too_long.sort_values("prompt_tokens", ascending=False).head(10))
        if not ALLOW_PROMPT_TRUNCATION:
            raise ValueError(
                f"{len(too_long)} prompts exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
                "Increase MAX_SEQ_LENGTH or shorten the prompt. vLLM truncation is intentionally not hidden here."
            )
    return summary


prompt_length_summary = audit_prompt_lengths(source_rows)


## 9. Generate Raw Rationales

In [ ]:
@dataclass
class RationaleRecord:
    """One raw sampled completion plus all fields needed for filtering diagnostics.

    Fields:
        id/family/gold_answer: Source row identity and target answer.
        candidate_index: Which sample attempt this was for that row.
        extracted_answer/is_correct: Answer parsed from raw_output and strict correctness flag.
        gate_ok/gate_reason: Non-answer quality gate result and reason.
        generated_tokens/hit_max_new_tokens/seconds: Generation diagnostics.
        raw_output: Full sampled completion saved before filtering.
    """
    id: str
    family: str
    candidate_index: int
    gold_answer: str
    extracted_answer: str
    is_correct: bool
    gate_ok: bool
    gate_reason: str
    generated_tokens: int
    hit_max_new_tokens: bool
    seconds: float
    raw_output: str


def make_sampling_params(max_tokens: int, seed_offset: int = 0) -> SamplingParams:
    """Create vLLM sampling parameters for one answer-conditioned rationale pass."""
    return SamplingParams(
        n=1,
        max_tokens=max_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        seed=RANDOM_SEED + seed_offset,
        stop=["}"],
        include_stop_str_in_output=True,
    )


def generated_token_count(vllm_completion) -> int:
    """Return generated-token count from a vLLM completion across supported versions."""
    token_ids = getattr(vllm_completion, "token_ids", None)
    if token_ids is not None:
        return len(token_ids)
    text = getattr(vllm_completion, "text", "") or ""
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def finish_reason_is_length(vllm_completion, token_count: int, max_tokens: int) -> bool:
    """Detect max-token hits without depending on one exact vLLM finish_reason spelling."""
    finish_reason = str(getattr(vllm_completion, "finish_reason", "")).lower()
    if finish_reason in {"length", "max_tokens"}:
        return True
    return token_count >= max_tokens and finish_reason not in {"stop", "eos", "finished"}


def run_vllm_generation(prompts: list[str], max_tokens: int, seed_offset: int = 0) -> tuple[list, float]:
    """Generate prompt completions through vLLM and return outputs plus wall time."""
    start = time.time()
    outputs = llm.generate(prompts, make_sampling_params(max_tokens, seed_offset), use_tqdm=False)
    return outputs, time.time() - start


def benchmark_generation_speed(rows: pd.DataFrame) -> pd.DataFrame:
    """Run a short vLLM throughput probe and save it before the full rationale loop."""
    if not RUN_GENERATION_SPEED_BENCHMARK or rows.empty:
        return pd.DataFrame()
    bench_rows = rows.head(min(BENCHMARK_ROWS, len(rows))).copy()
    prompts = [build_candidate_prompt(row.question, row.family, row.gold_answer) for row in bench_rows.itertuples(index=False)]
    outputs, seconds = run_vllm_generation(prompts, BENCHMARK_MAX_NEW_TOKENS, seed_offset=-1)
    generated_tokens = []
    finish_reasons = []
    for output in outputs:
        completion = output.outputs[0]
        count = generated_token_count(completion)
        generated_tokens.append(count)
        finish_reasons.append(str(getattr(completion, "finish_reason", "")))
    total_tokens = sum(generated_tokens)
    bench = pd.DataFrame([{
        "backend": GENERATION_BACKEND,
        "rows": len(bench_rows),
        "max_new_tokens": BENCHMARK_MAX_NEW_TOKENS,
        "seconds": seconds,
        "rows_per_second": len(bench_rows) / max(seconds, 1e-9),
        "generated_tokens": total_tokens,
        "generated_tokens_per_second": total_tokens / max(seconds, 1e-9),
        "finish_reasons": ",".join(sorted(set(finish_reasons))),
    }])
    bench.to_csv(GENERATION_SPEED_BENCHMARK_CSV, index=False)
    print("wrote", GENERATION_SPEED_BENCHMARK_CSV)
    display(bench)
    return bench


generation_speed_benchmark = benchmark_generation_speed(source_rows)


def generate_candidate_batch(batch: pd.DataFrame, candidate_index: int) -> list[RationaleRecord]:
    """Sample one answer-conditioned rationale for each row in a batch and record raw/filter fields."""
    prompts = [build_candidate_prompt(row.question, row.family, row.gold_answer) for row in batch.itertuples(index=False)]
    outputs, batch_seconds = run_vllm_generation(prompts, MAX_NEW_TOKENS, seed_offset=candidate_index)
    records = []
    seconds_per_row = batch_seconds / max(1, len(batch))
    for row, output in zip(batch.itertuples(index=False), outputs):
        completion = output.outputs[0]
        raw_output = (completion.text or "").strip()
        generated_tokens = generated_token_count(completion)
        hit_max_new_tokens = finish_reason_is_length(completion, generated_tokens, MAX_NEW_TOKENS)
        extracted = extract_final_answer(raw_output)
        gate_ok, gate_reason = quality_gate(raw_output, hit_max_new_tokens, row.family)
        records.append(
            RationaleRecord(
                id=row.id,
                family=row.family,
                candidate_index=candidate_index,
                gold_answer=row.gold_answer,
                extracted_answer=extracted,
                is_correct=verify_answer(row.gold_answer, extracted),
                gate_ok=gate_ok,
                gate_reason=gate_reason,
                generated_tokens=generated_tokens,
                hit_max_new_tokens=hit_max_new_tokens,
                seconds=seconds_per_row,
                raw_output=raw_output,
            )
        )
    return records


existing_raw = pd.DataFrame()
done_pairs = set()
if RAW_CANDIDATES_CSV.exists():
    existing_raw = pd.read_csv(RAW_CANDIDATES_CSV, dtype=str).fillna("")
    if {"id", "candidate_index"}.issubset(existing_raw.columns):
        existing_raw = existing_raw[existing_raw["family"].isin(TARGET_SAMPLE_FAMILIES)].copy()
        existing_raw["candidate_index"] = existing_raw["candidate_index"].astype(int)
        done_pairs = set(zip(existing_raw["id"].astype(str), existing_raw["candidate_index"].astype(int)))
        print("resuming from existing hard-family raw rationales:", existing_raw.shape, "done pairs:", len(done_pairs))
    else:
        print("existing raw rationale file has unexpected columns; regenerating from scratch")
        existing_raw = pd.DataFrame()

new_records = []
for candidate_index in range(CANDIDATES_PER_ROW):
    print(f"rationale pass {candidate_index + 1}/{CANDIDATES_PER_ROW}", flush=True)
    pending = source_rows[
        ~source_rows["id"].astype(str).map(lambda row_id: (row_id, candidate_index) in done_pairs)
    ].reset_index(drop=True)
    print("pending rows:", len(pending))
    for start in tqdm(range(0, len(pending), GENERATION_BATCH_SIZE), unit="batch"):
        batch = pending.iloc[start:start + GENERATION_BATCH_SIZE].reset_index(drop=True)
        batch_records = generate_candidate_batch(batch, candidate_index)
        new_records.extend(batch_records)
        done_pairs.update((record.id, record.candidate_index) for record in batch_records)
        batch_number = (start // GENERATION_BATCH_SIZE) + 1
        if SAVE_RAW_EVERY_BATCHES and batch_number % int(SAVE_RAW_EVERY_BATCHES) == 0:
            snapshot = pd.concat(
                [existing_raw, pd.DataFrame([asdict(record) for record in new_records])],
                ignore_index=True,
            )
            snapshot = snapshot.drop_duplicates(["id", "candidate_index"], keep="last")
            snapshot.to_csv(RAW_CANDIDATES_CSV, index=False)
            print("checkpoint raw rationales:", RAW_CANDIDATES_CSV, snapshot.shape, flush=True)

raw_candidates = pd.concat(
    [existing_raw, pd.DataFrame([asdict(record) for record in new_records])],
    ignore_index=True,
)
if not raw_candidates.empty:
    raw_candidates = raw_candidates[raw_candidates["family"].isin(TARGET_SAMPLE_FAMILIES)].copy()
    raw_candidates = raw_candidates.drop_duplicates(["id", "candidate_index"], keep="last")
    raw_candidates["candidate_index"] = raw_candidates["candidate_index"].astype(int)
    raw_candidates["generated_tokens"] = pd.to_numeric(raw_candidates["generated_tokens"], errors="coerce").fillna(0).astype(int)
    raw_candidates["seconds"] = pd.to_numeric(raw_candidates["seconds"], errors="coerce").fillna(0.0)
    for bool_col in ["is_correct", "gate_ok", "hit_max_new_tokens"]:
        raw_candidates[bool_col] = raw_candidates[bool_col].astype(str).str.lower().isin(["true", "1", "yes"])
raw_candidates.to_csv(RAW_CANDIDATES_CSV, index=False)
print("wrote", RAW_CANDIDATES_CSV, raw_candidates.shape)
display(raw_candidates.head())


## 10. Accept, Select, And Diagnose

In [ ]:
accepted = raw_candidates[(raw_candidates["is_correct"]) & (raw_candidates["gate_ok"])].copy()
accepted["raw_chars"] = accepted["raw_output"].str.len()
accepted["trace_signal_score"] = accepted.apply(lambda row: trace_signal_score(row.raw_output, row.family), axis=1)
accepted = accepted.sort_values(
    ["family", "trace_signal_score", "raw_chars", "generated_tokens", "candidate_index", "id"],
    ascending=[True, False, True, True, True, True],
)

if MAX_ACCEPTED_PER_FAMILY is not None:
    accepted = accepted.groupby("family", group_keys=False).head(int(MAX_ACCEPTED_PER_FAMILY))

selected = accepted.sort_values(
    ["id", "trace_signal_score", "raw_chars", "generated_tokens", "candidate_index"],
    ascending=[True, False, True, True, True],
).drop_duplicates("id", keep="first")

accepted.to_csv(ACCEPTED_CANDIDATES_CSV, index=False)

summary = []
for family, fam_source in source_rows.groupby("family"):
    fam_raw = raw_candidates[raw_candidates["family"] == family]
    fam_accepted = accepted[accepted["family"] == family]
    fam_selected = selected[selected["family"] == family]
    summary.append({
        "family": family,
        "source_rows": len(fam_source),
        "raw_candidates": len(fam_raw),
        "correct_candidates": int(fam_raw["is_correct"].sum()),
        "accepted_candidates": len(fam_accepted),
        "selected_rows": len(fam_selected),
        "row_acceptance_rate": len(fam_selected) / max(1, len(fam_source)),
        "candidate_acceptance_rate": len(fam_accepted) / max(1, len(fam_raw)),
        "max_token_hits": int(fam_raw["hit_max_new_tokens"].sum()),
    })
summary_df = pd.DataFrame(summary).sort_values("family")
summary_df.to_csv(ACCEPTANCE_SUMMARY_CSV, index=False)

print("accepted candidates:", accepted.shape)
print("selected rows:", selected.shape)
display(summary_df)
display(raw_candidates.groupby(["family", "gate_reason"]).size().reset_index(name="count"))
display(selected[["id", "family", "gold_answer", "extracted_answer", "generated_tokens", "raw_output"]].head(10))


## 11. Write Trainable Trace CSV

In [ ]:
selected_trace = selected[["id", "raw_output"]].rename(columns={"raw_output": "trace"})
dataset = source_rows[["id", "question", "gold_answer", "family"]].merge(selected_trace, on="id", how="left")

if not set(dataset["family"].unique()).issubset(TARGET_SAMPLE_FAMILIES):
    raise ValueError("Trace dataset contains non-target families; notebook 12 should output only cipher/bit STaR rationales")

if INCLUDE_BOXED_FALLBACKS:
    dataset["trace"] = dataset.apply(
        lambda row: row.trace if isinstance(row.trace, str) and row.trace.strip() else boxed_only_trace(row.gold_answer),
        axis=1,
    )
else:
    dataset = dataset[dataset["trace"].fillna("").str.strip().ne("")].copy()

dataset["boxed_count"] = dataset["trace"].str.count(re.escape("\\boxed{"))
bad_boxed = dataset[dataset["boxed_count"] != 1]
if not bad_boxed.empty:
    display(bad_boxed[["id", "family", "boxed_count", "trace"]].head())
    raise ValueError("Every training trace must contain exactly one boxed answer")

dataset["text_after_box"] = dataset["trace"].map(lambda text: extract_last_boxed_content_and_tail(text)[1].strip())
bad_tail = dataset[dataset["text_after_box"].ne("")]
if not bad_tail.empty:
    display(bad_tail[["id", "family", "text_after_box", "trace"]].head())
    raise ValueError("Some selected traces have text after the final boxed answer")

dataset["extracted_answer"] = dataset["trace"].map(extract_final_answer)
dataset["verified"] = dataset.apply(lambda row: verify_answer(row.gold_answer, row.extracted_answer), axis=1)
if not bool(dataset["verified"].all()):
    display(dataset[~dataset["verified"]][["id", "family", "gold_answer", "extracted_answer", "trace"]].head())
    raise ValueError("Some selected traces do not verify against gold answers")

trace_dataset = dataset[["id", "question", "trace", "gold_answer"]].copy()
trace_dataset.to_csv(TRACE_DATASET_CSV, index=False)

print("wrote trainable STaR rationale dataset:", TRACE_DATASET_CSV)
print("rows:", trace_dataset.shape[0])
display(dataset["family"].value_counts().rename_axis("family").reset_index(name="rows"))
display(trace_dataset.head())


## 12. Save Run Config

In [ ]:
run_config = {
    "experiment_name": EXPERIMENT_NAME,
    "method": "answer_conditioned_star_rationale_hard_families_vllm",
    "model_name": MODEL_NAME,
    "train_csv_path": str(TRAIN_CSV_PATH),
    "trace_dataset_csv": str(TRACE_DATASET_CSV),
    "target_sample_families": sorted(TARGET_SAMPLE_FAMILIES),
    "probe_only": PROBE_ONLY,
    "probe_total_rows": PROBE_TOTAL_ROWS,
    "probe_rows_per_family_cap": PROBE_ROWS_PER_FAMILY_CAP,
    "row_limit": ROW_LIMIT,
    "rows_per_family_cap": ROWS_PER_FAMILY_CAP,
    "candidates_per_row": CANDIDATES_PER_ROW,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "save_raw_every_batches": SAVE_RAW_EVERY_BATCHES,
    "generation_backend": GENERATION_BACKEND,
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "vllm_dtype": VLLM_DTYPE,
    "vllm_tensor_parallel_size": VLLM_TENSOR_PARALLEL_SIZE,
    "vllm_gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
    "vllm_enforce_eager": VLLM_ENFORCE_EAGER,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "repetition_penalty": REPETITION_PENALTY,
    "run_generation_speed_benchmark": RUN_GENERATION_SPEED_BENCHMARK,
    "benchmark_rows": BENCHMARK_ROWS,
    "benchmark_max_new_tokens": BENCHMARK_MAX_NEW_TOKENS,
    "require_exactly_one_box": REQUIRE_EXACTLY_ONE_BOX,
    "reject_max_token_hits": REJECT_MAX_TOKEN_HITS,
    "reject_text_after_box": REJECT_TEXT_AFTER_BOX,
    "require_family_trace_signal": REQUIRE_FAMILY_TRACE_SIGNAL,
    "max_accepted_chars": MAX_ACCEPTED_CHARS,
    "max_accepted_per_family": MAX_ACCEPTED_PER_FAMILY,
    "include_boxed_fallbacks": INCLUDE_BOXED_FALLBACKS,
    "random_seed": RANDOM_SEED,
    "outputs": {
        "raw_rationales_csv": str(RAW_CANDIDATES_CSV),
        "accepted_rationales_csv": str(ACCEPTED_CANDIDATES_CSV),
        "rationale_summary_csv": str(ACCEPTANCE_SUMMARY_CSV),
        "trace_dataset_csv": str(TRACE_DATASET_CSV),
        "generation_speed_benchmark_csv": str(GENERATION_SPEED_BENCHMARK_CSV),
    },
}

RUN_CONFIG_PATH.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("wrote", RUN_CONFIG_PATH)


## 13. Training Handoff

Do not train from this probe automatically. First inspect the accepted rationales by family. If the 32-row probe is compact and mechanically plausible, rerun the notebook with:

```python
PROBE_ONLY = False
TARGET_SAMPLE_FAMILIES = {"cipher", "bit_manipulation"}
ROWS_PER_FAMILY_CAP = None
CANDIDATES_PER_ROW = 4
TEMPERATURE = 0.5
```

Then mix the generated hard-family CSV into an existing training notebook with deterministic/easy-family rehearsal.

Recommended first training setup after a successful full build:

```python
EXPERIMENT_NAME = "exp20_star_hard_mix_r4_ep1"
TRACE_TARGET_MODE = "mixed_deterministic_plus_star_hard"
TRACE_TRAINING_CSV_CANDIDATES = [
    Path("/content/trace_exp19_cipher_bit_star_rationale_probe_v4.csv"),
    Path("/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts/outputs/exp19_cipher_bit_star_rationale_probe_v4/trace_exp19_cipher_bit_star_rationale_probe_v4.csv"),
]
```

Keep generated eval at 64 rows for the first adapter and inspect cipher/bit raw completions before any Kaggle submission.

## Occam Audit Before Training

- If the probe rationales mention that the answer was provided, tighten the prompt or reject those traces.
- If traces hit max tokens, keep `MAX_NEW_TOKENS=256`, shorten the prompt, and keep the stop-after-box sampling guard active.
- If cipher rationales are just restating the answer without mappings, switch to constrained candidate-list or multiple-choice selection.
- If bit rationales invent unverified rules, use programmatic DSL/rule search instead.
- Do not add DPO, online RL, or multi-stage loops until this single answer-conditioned dataset produces evidence.